# Tiremo® Accelerator Workshops'a Hoş Geldiniz!

Empa Electronics tarafından düzenlenen Tiremo® Accelerator Workshops etkinliğine hoş geldiniz. Bu açık kaynaklı repository, workshop etkinliğimizde deneyimleyeceğiniz "Tiremo®Cortex İçin Edge-AI Çözümleri Geliştirme" uygulaması çalışma ortamını edinebilmeniz ve aktivitelere kolaylıkla eşlik edebilmeniz için sizinle paylaşılmıştır.

Bu script, bir uçta yapay zeka çözümünün geliştirilme adımlarının gösterimi için oluşturulmuştur.

**Uygulama Adımları:**

1. Gereksinimlerin Dahil Edilmesi (Requirements)

2. Veri Ön-İşleme

3. Yapay Zeka Modeli Oluşturma

4. Model Eğitimi & Çözüm Geliştirme (Development)

5. Modeli Dışa Aktarma & Dağıtım (Deployment)

## 0. Kurulum (Bulut Çalışma Ortamı)
Bu başlık, bulut çalışma ortamı için gerekli kurulumların otomatik olarak yapılması için oluşturulmuştur.  
Kurulum başlığı altındaki kod hücrelerini **YALNIZCA** Google Colab ortamı için çalıştırınız.

1. Python versiyonu olarak tercih edilen 3.10 versiyonunun kurulu olduğunu teyit ediniz.

In [ ]:
!python3 --version

2. Gerekli klasör ve kaynak dosyalarını oluşturunuz/indiriniz.

In [ ]:
!mkdir -p Datasets && mkdir -p Models

!wget https://raw.githubusercontent.com/Empa-Electronics/Tiremo-Accelerator-Workshops-AIoT/refs/heads/master/Activity2_EdgeAI_Solutions_and_Deployment/Datasets/Dataset_HMR_3Axis_EmpaElectronics.csv -O Datasets/Dataset_HMR_3Axis_EmpaElectronics.csv

3. Gerekli Python modüllerini edininiz. Bu hücrenin ilk çalıştırılması sonrasında "Restart Session" yaparak Colab Runtime'ı güncelleyiniz.

In [ ]:
!pip3 install -r https://raw.githubusercontent.com/Empa-Electronics/Tiremo-Accelerator-Workshops-AIoT/refs/heads/master/Activity2_EdgeAI_Solutions_and_Deployment/requirements.txt

## 1. Gereksinimlerin Dahil Edilmesi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import floor
from scipy.stats import mode

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

import optuna
import emlearn

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# dataset path
path_dataset = "./Datasets/Dataset_HMR_3Axis_EmpaElectronics.csv"
# class ID to class name mapping
classes = {0: "CIRCLE",
           1: "HORIZONTAL",
           2: "STANDBY",
           3: "TRIANGLE",
           4: "VERTICAL"}

## 2. Veri Ön-İşleme
### 2.1. Ham Veri Eldesi

In [ ]:
df_dataset = pd.read_csv(path_dataset)
df_dataset

### 2.2. "Feature" & "Labels" Ayrıştırımı

In [ ]:
df_feats, df_labels = df_dataset.drop(columns=["class"]), pd.DataFrame(df_dataset["class"])

Veri seti "feature" sütunları:

In [ ]:
df_feats

Veri seti "label" sütunu:

In [ ]:
df_labels

Feature ve Label dataframe yapıları için "null" değer kontrolü:

In [ ]:
print(f"[features] Number of NAs: {df_feats.isna().sum().sum()}")
print(f"[features] Number of nulls: {df_feats.isnull().sum().sum()}")
print(f"[labels] Number of NAs: {df_labels.isna().sum().sum()}")
print(f"[labels] Number of nulls: {df_labels.isnull().sum().item()}")

Veri seti sınıflarının listesi:

In [ ]:
list_categories = np.array(sorted(set(df_labels.to_numpy().flatten()))).reshape(-1, 1)
list_categories

### 2.4. Veri Setinden Sekans Paketleri (Sequence Batchs) Oluşturulması
Sabit bir sekans uzunluğu (sequence lenght) ve örtüşme oranı (overlapping ratio) tanımlaması:  
  
Sekans uzunluğu, bir tablo veri seti içerisindeki örneklerin (örneğin: sensör verisi kaydı) tekil olarak değerlendirilmesi yerine bir pencere/sekans örneği içerisinde veri kesitleri olarak ifade edilmesinde kullanılan bir sabittir.  
  
Örnek:   
Başlangıçta 1000 veri örneği ve 6 feature'dan oluşan bir veri seti için (_veri seti boyutu: 1000 x 6_)  
"sekans uzunluğu = 200 örnek" olarak seçilir ve veri setine uygulanırsa

5 adet 200 örnekli sekans paketi elde edilmiş olur (_veri seti yeni boyutu: 5 x 200 x 6_).

In [ ]:
seq_length = 128
overlapping_ratio = 0.33

Sekans paketleri oluşturma fonksiyonunun tanımlanması

In [ ]:
def create_sequences(data, labels, num_samples=128, overlap=0.5):
    """Takes tabular data and creates times sequences with given sample width."""

    # create empty lists for stacking
    data_sequences = []
    data_labels = []
    # get the number of examples
    num_examples = data.shape[0]
    # compute stride value
    strides = num_samples - floor(num_samples * overlap)
    # compute the number of sequences to use as iterator
    num_sequences = floor((num_examples - num_samples) / strides) + 1

    # iterate for sequence range
    for ind in range(num_sequences):

        # define start index
        ind_start = ind * strides
        # define end index
        ind_end = ind_start + num_samples
        # get the current data slice by using start and end indexes
        slice_seq_acc = data.values[ind_start:ind_end]
        # get the current labels slice by using start and end indexes
        slice_seq_label = labels.values[ind_start:ind_end]

        # take the modal value of label slice: (replace with .mode())
        label_seq = mode(slice_seq_label, keepdims=True)[0][0]

        # stack current slices
        data_sequences.append(slice_seq_acc)
        data_labels.append(label_seq)

    # convert stacks to numpy array
    data_sequences = np.array(data_sequences)
    data_labels = np.array(data_labels).ravel()

    # return sequence and label stacks as X and Y
    return data_sequences, data_labels

Orijinal veri setinden sekans paketleri içeren veri setinin eldesi:

In [ ]:
x_feats_seq, y_labels_seq = create_sequences(
                                    data=df_feats,
                                    labels=df_labels,
                                    num_samples=seq_length,
                                    overlap=overlapping_ratio)
print("Shape of Sequence Features:", x_feats_seq.shape)
print("Shape of Sequence Labels:", y_labels_seq.shape)

(n, 128, 3) -> (m, 1, 384)

In [ ]:
x_feats_seq = x_feats_seq.reshape(x_feats_seq.shape[0], -1)

### 2.5. Train and Test Veri Setlerinin Ayrımı
Train/Test bölme oranı tanımlanması:

In [ ]:
test_split_ratio = 0.1

Veri setinin bölünmesi:

In [ ]:
x_train_seq, x_test_seq, y_train_seq, y_test_seq = train_test_split(x_feats_seq, y_labels_seq, test_size=test_split_ratio, shuffle=True, random_state=RANDOM_SEED)
print(f"Shapes of Train Set - Features: {x_train_seq.shape} - Labels: {y_train_seq.shape}")
print(f"Shapes of Test Set - Features: {x_test_seq.shape} - Labels: {y_test_seq.shape} ")

## 3. Yapay Zeka Modelinin Oluşturulması
### 3.1. Hiperparametre Optimizasyonu
Random Forest Sınıflandırma modelinin Optuna ile hiperparametre optimizasyonu yapılarak oluşturulması:

In [ ]:
def objective(trial):

    # define hyperparameters to optimize
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)

    # create a model with given hyperparameters
    model = RandomForestClassifier(n_estimators=n_estimators,
                                   max_depth=max_depth,
                                   min_samples_split=min_samples_split,
                                   min_samples_leaf=min_samples_leaf,
                                   random_state=RANDOM_SEED)

    # fit the model with train set
    model.fit(x_train_seq, y_train_seq)

    # predict with test set
    y_pred_seq = model.predict(x_test_seq)

    # compute f1 score and return as objective metric
    f1 = f1_score(y_test_seq, y_pred_seq, average="weighted")
    return f1

### 3.2. Model Eğitimi

In [ ]:
# create an Optuna study and optimize the objective f1 score
study = optuna.create_study(study_name="Tiremo®Cortex HMR Hyperparameter Optimization", 
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
                            direction="maximize")
# start optimization
with np.errstate(under="ignore"):
    study.optimize(objective, n_trials=20)

# print findings for the best trial
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# train the final model with the best performed hyperparameters
model = RandomForestClassifier(**best_params, random_state=RANDOM_SEED)
model.fit(x_train_seq, y_train_seq)

### 3.3. Sonuçların Değerlendirilmesi
Eğitilmiş modelin performans metriklerinin ölçülmesi:

In [ ]:
y_preds = model.predict(x_test_seq)
model_f1score = f1_score(y_test_seq, y_preds, average='macro')
print(f"F1 Score of the Model: {(model_f1score*100):.2f}%")

Confusion Matrix'in incelenmesi:

In [ ]:
# plotting the confusion matrix
plt.style.use("default")
y_pred_named = np.vectorize(classes.get)(y_preds)
y_test_named = np.vectorize(classes.get)(y_test_seq)
ConfusionMatrixDisplay.from_predictions(y_test_named, y_pred_named)
plt.xticks(rotation=90)
plt.show()

## 4. Eğitilmiş Modelin Dışa Aktarımı
Geliştirme ortamı belleğinde tutulan yapay zeka modelinin (mimari + öğrenilmiş parametreler) çözüme dönüştürülmesi için statik bir dosya halinde saklanması gerekmektedir.

In [ ]:
def convert_tree(input_model, output_path):
    import emlearn
    import os

    # save the converted model
    edge_model = emlearn.convert(input_model, 
                                 method='inline', 
                                 dtype='int'
                                 )
    # create output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)
    # save the model to a header file
    edge_model.save(
        file=os.path.join(output_path, 'Tiremo_model.h'), 
        name='Tiremo_model'
        )

In [ ]:
convert_tree(model, output_path="./Models")

*Models klasöründe **Tiremo_model.h** ismiyle kaydettiğimiz dosyayı bilgisayarınıza indirin ve Tiremo®Cortex üzerinde çalıştıracağınız uygulama içerisinde kullanmak üzere hazır bulundurun.*

## 5. Eğitilen Model İle Uçta Yapay Zeka Çözümü
_Eğitilen modelin Tiremo®Cortex üzerinde çözüm olarak dağıtılması adımı için Tiremo®Intelligence kapsamında geliştirilen proje adımlarıyla devam ediniz._